In [9]:
from pathlib import Path
from PIL import Image
from collections import Counter
import random
import pandas as pd

In [10]:
# Settings TODO: set the root(images) and label path, which are located on your pc
root = Path(r"\ExpW_extracted\origin")
label_path = Path(r"\label.lst")
SEED = 90051
MAX_PER_CLASS = 2600
random.seed(SEED)

In [3]:
# Read the label list and check the face sizes
eligible_rows = []
with open(label_path, "r", encoding="utf-8") as f:
    for line in f:
        num_of_columns = line.strip().split()
        if len(num_of_columns) != 8:
            continue #if the number of columns is not equal to eight, then the data is recognized as invalid 

        original_image_name = num_of_columns[0]
        face_id = num_of_columns[1]#Indicates which face it is in the image as a id
        top = int(float(num_of_columns[2]))
        left = int(float(num_of_columns[3]))
        right = int(float(num_of_columns[4]))
        bottom = int(float(num_of_columns[5]))#Score representing the confidence level of face detection
        confidence = float(num_of_columns[6])#Confidence level of element detection
        correct_facial_class = int(num_of_columns[7])#Correct facial expression class

        img_path = root / original_image_name
        if not img_path.exists():
            continue

        with Image.open(img_path) as img:
            img_width, img_height = img.size

        # check the bbox is inside the image, if not then adjust the image size 
        if left < 0:left = 0
        if top < 0:top = 0
        if right > img_width:right = img_width
        if bottom > img_height:bottom = img_height
        width = right - left
        height = bottom - top

        # Check whether the face images are over the required size(100x100)
        if width >= 100 and height >= 100:
            eligible_rows.append({"original_image_name": original_image_name, "face_id": face_id,
                "top": top, "left": left, "right": right, "bottom": bottom, "width": width, "height": height,
                "confidence": confidence, "correct_facial_class": correct_facial_class})
print("Eligible face samples:", len(eligible_rows))

Eligible face samples: 41746


In [4]:
# Keep one face per source image(if there are over one fafe in a image, then only highest confident face is extracted from one picture)
best_face_per_image = {}
for row in eligible_rows:
    original_image_name = row["original_image_name"]
    if (original_image_name not in best_face_per_image or 
        row["confidence"] > best_face_per_image[original_image_name]["confidence"]):
        best_face_per_image[original_image_name] = row
unique_rows = list(best_face_per_image.values())
print("The number of facial images that remained in the end:", len(unique_rows))

The number of facial images that remained in the end: 36264


In [5]:
# check class distribution
emotion_names = {0: "angry", 1: "disgust", 2: "fear", 3: "happy", 4: "sad", 5: "surprise", 6: "neutral"}
class_counts = Counter(row["correct_facial_class"] for row in unique_rows)
print("Ficial class counts before sampling:")
for correct_facial_class in range(7):
    print(correct_facial_class,":",emotion_names[correct_facial_class],"->",class_counts[correct_facial_class])

Ficial class counts before sampling:
0 : angry -> 2179
1 : disgust -> 1765
2 : fear -> 591
3 : happy -> 9668
4 : sad -> 5796
5 : surprise -> 3634
6 : neutral -> 12631


In [6]:
#Downsample majority classes(Max size is 2600 per ficial class) to prevent data bias
selected_rows = []
for correct_facial_class in range(7):
    class_rows = [row for row in unique_rows if row["correct_facial_class"] == correct_facial_class]
    if len(class_rows) > MAX_PER_CLASS:
        class_rows = random.sample(class_rows,MAX_PER_CLASS)#randomly re-extract the data to reach the upper limit of 2600 entries
    selected_rows.extend(class_rows)

In [7]:
#Create final subset metadata for the research to prepare saving a file as a csv file
file_sampling_dataset = pd.DataFrame(selected_rows)
file_sampling_dataset = file_sampling_dataset.sort_values(["correct_facial_class", "original_image_name"]).reset_index(drop=True)
file_sampling_dataset.insert(0,"sample_face_id",[f"expw_{i:05d}" for i in range(len(file_sampling_dataset))])
print("Final subset size:", len(file_sampling_dataset))
print("Unique images:", file_sampling_dataset["original_image_name"].nunique())
print("\nFinal facial class distributions:")
final_counts = Counter(file_sampling_dataset["correct_facial_class"])
for correct_facial_class in range(7): 
    print(correct_facial_class,":",emotion_names[correct_facial_class],"->",final_counts[correct_facial_class])

Final subset size: 14935
Unique images: 14935

Final facial class distributions:
0 : angry -> 2179
1 : disgust -> 1765
2 : fear -> 591
3 : happy -> 2600
4 : sad -> 2600
5 : surprise -> 2600
6 : neutral -> 2600


In [8]:
#Save metadata subset to csv 
file_sampling_dataset.to_csv("expw_subset_metadata_comp90051.csv",index=False)